In [ ]:
from IPython.display import Markdown
import torch
import string
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## data

In [ ]:
with open("../data/usnames.txt", "r") as f:
    names = f.readlines()

In [ ]:
Markdown(f"Number of names = {len(names)}")

In [ ]:
names = [n.strip() for n in names]

In [ ]:
names[0]

In [ ]:
'\'' + names[np.argmax(np.array([len(n) for n in names])).item()] + '\' is the longest name'

In [ ]:
'\'' + names[np.argmin(np.array([len(n) for n in names])).item()] + '\' is the shortest name'

## tokenize

In [ ]:
tok2idx = {c: i for i, c in enumerate('.' + string.ascii_lowercase)}
idx2tok = {i: c for c, i in tok2idx.items()}

In [ ]:
tok2idx

In [ ]:
[(x, y) for x, y in zip('.emma', 'emma.')]

## creating a count matrix

In [ ]:
C = torch.zeros((len(tok2idx), len(tok2idx)))

for name in names:
    for c, nc in zip('.' + name, name+'.'):
        C[tok2idx[c]][tok2idx[nc]] += 1

In [ ]:
plt.figure(figsize=(16, 16)); plt.imshow(C, cmap="Blues")

for y in range(len(tok2idx)):
    for x in range(len(tok2idx)):
        label = idx2tok[y]+idx2tok[x]
        plt.text(x, y, label, ha="center", va="top", color="gray")
        plt.text(x, y, int(C[y][x]), ha="center", va="bottom", color="gray")
plt.axis('off');

In [ ]:
P = C.float() / C.sum(axis=1, keepdim=True)

In [ ]:
plt.figure(figsize=(16, 16)); plt.imshow(C, cmap="Blues")

for y in range(len(tok2idx)):
    for x in range(len(tok2idx)):
        label = idx2tok[y]+idx2tok[x]
        plt.text(x, y, label, ha="center", va="top", color="gray")
        plt.text(x, y, f"{P[y][x]:.2f}", ha="center", va="bottom", color="gray")
plt.axis('off');

## creating a predictor

In [ ]:
g = torch.Generator().manual_seed(2147483647)

for i in range(10):
    ix = 0
    while True: 
        ix = torch.multinomial(P[ix], num_samples=1, replacement=True, generator=g).item()
        print(idx2tok[ix], end="")
        if ix==0: break
    print("")

## log loss

how can we summarize the quality of the model?